# Contract-to-Test: schema and SQL to coverage witnesses

This notebook creates fully synthetic local Pandas data. It does not read production rows or execute the supplied SQL. The SQL is parsed into a bounded, documented workload contract.


## Prerequisites

```bash
pip install great-generator
```

Phase 1 supports Pandas, acyclic simple relationships, equality joins, and the documented predicate subset. Spark-native witness materialization, cycles, and full-query equivalence are planned rather than simulated here.


In [ ]:
from great_generator.testing import compile_test_dataset

tables = {
    "dim_customer": {
        "schema": "customer_id int primary key, region string not null",
        "rows": 30,
    },
    "fact_orders": {
        "schema": (
            "order_id int primary key, "
            "customer_id int references dim_customer.customer_id, "
            "order_status string not null, order_date date not null, amount double"
        ),
        "rows": 60,
    },
}

query = """
SELECT c.region, COUNT(*) AS order_count
FROM fact_orders f
JOIN dim_customer c ON f.customer_id = c.customer_id
WHERE c.region = 'EAST'
  AND f.order_status = 'COMPLETED'
  AND f.order_date >= DATE '2026-01-01'
GROUP BY c.region
"""


In [ ]:
result = compile_test_dataset(
    schema=tables,
    queries=[query],
    coverage=["predicates", "joins", "boundaries"],
    seed=42,
)

result.datasets.keys()


In [ ]:
result.datasets["fact_orders"].head(8)


In [ ]:
validation = result.validate()
assert validation["passed"]
validation


In [ ]:
coverage = result.coverage_report()
coverage


## Expected bounded outcomes

These assertions come from a separate SQL three-valued predicate evaluator. They validate the allocated witness rows; they are not a claim of arbitrary full-query equivalence.


In [ ]:
assert all(item["verified"] for item in result.expected_results)
result.expected_results


## Optional export

Uncomment to write table files plus the workload contract, scenarios, reports, expected assertions, and reproducibility manifest. Existing non-empty directories are rejected unless `overwrite=True`.


In [ ]:
# result.export("./output/contract_test", format="parquet")
# result.generate_report(format="html", output="./output/contract_test/report.html")
